# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and column IDs. All `@id` fields are used for referencing the entities.

In [ ]:
# List available record sets by their @id
record_sets = dataset.metadata.record_sets
print(f"Available record sets (by @id):\n")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")

# Inspect fields and columns in each record set
overview = {}
for rs in record_sets:
    print(f"\nRecordSet: {rs['name']} (@id: {rs['@id']})")
    fields = rs.get('field', [])
    print(f"  Fields (@id):")
    for f in fields:
        # f can be dict or just @id string
        if isinstance(f, dict):
            print(f"    - {f['@id']} (name: {f.get('name', 'N/A')})")
        else:
            print(f"    - {f}")
    columns = rs.get('column', [])
    if columns:
        print(f"  Columns (@id):")
        for c in columns:
            if isinstance(c, dict):
                print(f"    - {c['@id']} (name: {c.get('name', 'N/A')})")
            else:
                print(f"    - {c}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s are referenced explicitly.

In [ ]:
# Build a list of record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

print(f"Extracting data for record sets:")
for rsid in record_set_ids:
    print(f"  - {rsid}")
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"    -> Loaded {df.shape[0]} records, columns: {df.columns.tolist()}")
    else:
        print(f"    -> No records found for this record set.")

# Show a sample from the first record set with data
first_rs_with_data = None
for rsid in record_set_ids:
    if rsid in dataframes:
        first_rs_with_data = rsid
        break
if first_rs_with_data:
    print(f"\nColumns in record set '@id': {first_rs_with_data}")
    print(dataframes[first_rs_with_data].columns.tolist())
    dataframes[first_rs_with_data].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a field, normalizing numeric fields, and grouping data by a chosen field. All manipulations use field `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For demonstration, select the first available record set and numeric-looking field
# You may customize these variables using the output of the previous cell
record_set_id = first_rs_with_data
df = dataframes[record_set_id]

# Inspect and select a numeric field by @id (e.g., "peptide_count" or similar)
print(f"Examining columns for numeric selection:")
print(df.columns)

# Attempt to pick a field with int/float values
numeric_field = None
for col in df.columns:
    # Try to parse data as float to check if numeric
    try:
        vals = pd.to_numeric(df[col].dropna().head(10))
        if vals.dtype.kind in 'if' and (vals.max() > 0):
            numeric_field = col
            break
    except Exception:
        continue
if not numeric_field:
    print("No numeric field detected for EDA.")
else:
    print(f"Selected numeric field by @id: {numeric_field}")
    # Convert column to numeric (in case of strings)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()
    # Filter: keep only values above threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records in '{record_set_id}' with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records (z-score):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to pick a group field (categorical, with a small set of unique values)
    group_field = None
    for col in df.columns:
        if col == numeric_field:
            continue
        nunique = df[col].nunique()
        if 2 < nunique < 20:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping data by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships for numeric and categorical fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded Croissant-described dataset metadata and explored all available record sets and analytical fields using their `@id` identifiers.
- Loaded records from the FAIR² dataset using `mlcroissant` without assuming a fixed data schema.
- Applied typical data cleaning and EDA steps, including numeric field selection, filtering, normalization, grouping, and simple visualizations.

**Next steps:**
- Dive deeper into domain-specific analyses, such as examining protein abundances or modifications by sample type or condition.
- Use entity `@id` fields throughout to ensure traceability and robust data pipeline construction.
- Combine Croissant metadata with external semantic or experimental context for advanced analytics.